In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('dark_background')
sns.set_palette('viridis')
PURPLE = '#7c3aed'

print('All imports OK ✓')

In [ ]:
df = pd.read_csv('../data/tiktok_retention_data.csv', parse_dates=['signup_date'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nNull values:', df.isnull().sum().sum())
print('\nChurn rate:', df['churned'].mean().round(3))
print('D1 retention:', df['d1_retained'].mean().round(3))
print('D7 retention:', df['d7_retained'].mean().round(3))
print('D30 retention:', df['d30_retained'].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment distribution
seg_counts = df['segment'].value_counts()
axes[0].bar(seg_counts.index, seg_counts.values, color=PURPLE, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0].set_title('User Distribution by Segment', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Users')
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=10, color='white')

# Channel distribution
ch_counts = df['acquisition_channel'].value_counts()
colors = ['#7c3aed','#06b6d4','#f59e0b','#10b981','#f87171']
axes[1].pie(ch_counts.values, labels=ch_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'color': 'white'})
axes[1].set_title('Acquisition Channel Mix', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('../data/seg_channel_overview.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
cohort = df.groupby('cohort_month').agg(
    size=('user_id','count'),
    d1=('d1_retained','mean'),
    d7=('d7_retained','mean'),
    d30=('d30_retained','mean'),
    churn=('churned','mean'),
).reset_index().sort_values('cohort_month')

heatmap_matrix = cohort.set_index('cohort_month')[['d1','d7','d30']].T * 100
heatmap_matrix.index = ['Day 1', 'Day 7', 'Day 30']

plt.figure(figsize=(18, 4))
sns.heatmap(heatmap_matrix, annot=True, fmt='.1f', cmap='Purples',
            linewidths=0.5, linecolor='#111', vmin=0, vmax=100,
            cbar_kws={'label': 'Retention %', 'shrink': 0.8})
plt.title('Monthly Cohort Retention Heatmap (%)', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/cohort_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

cohort

In [ ]:
seg_ret = df.groupby('segment')[['d1_retained','d7_retained','d30_retained']].mean() * 100
seg_ret.columns = ['Day 1','Day 7','Day 30']

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'Casual':'#7c3aed','Creator':'#06b6d4','Power User':'#10b981','Lurker':'#f59e0b'}

for seg in seg_ret.index:
    ax.plot(['Day 1','Day 7','Day 30'], seg_ret.loc[seg],
            marker='o', linewidth=2.5, markersize=8,
            label=seg, color=colors_map.get(seg, PURPLE))
    ax.annotate(f'{seg_ret.loc[seg, "Day 30"]:.1f}%',
                xy=('Day 30', seg_ret.loc[seg, 'Day 30']),
                xytext=(5, 0), textcoords='offset points', fontsize=9)

ax.set_title('Retention Curves by User Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Retention Rate (%)')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../data/retention_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
# Kaplan-Meier style survival estimate
# At each day t, survival = proportion of users still active (not yet churned)

churned_users = df[df['churned'] == 1]
days = np.arange(1, 91)

segments_list = df['segment'].unique()
fig, ax = plt.subplots(figsize=(12, 6))

for seg in segments_list:
    seg_df = df[df['segment'] == seg]
    survival = []
    for d in days:
        survived = (seg_df['days_until_churn'] > d).mean()
        survival.append(survived)
    ax.plot(days, survival, label=seg, linewidth=2, color=colors_map.get(seg, PURPLE))

ax.set_title('Survival Curves by Segment (Days Until Churn)', fontsize=14, fontweight='bold')
ax.set_xlabel('Day (post signup)')
ax.set_ylabel('Proportion Still Active')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
ax.grid(alpha=0.2)
ax.axvline(30, color='white', linestyle='--', alpha=0.3, label='D30')
plt.tight_layout()
plt.savefig('../data/survival_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
features = ['used_fyp','used_search','used_live','used_shop','used_duet','notification_enabled']
labels   = ['For You Page','Search','Live','Shop','Duet','Push Notifs']

rows = []
for feat, label in zip(features, labels):
    with_f    = df[df[feat]==1]['d30_retained'].mean() * 100
    without_f = df[df[feat]==0]['d30_retained'].mean() * 100
    rows.append({'Feature': label, 'With': with_f, 'Without': without_f, 'Lift': with_f - without_f})

impact = pd.DataFrame(rows).sort_values('Lift', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(impact))
w = 0.35
ax.bar(x - w/2, impact['Without'], w, label='Without Feature', color='#374151', edgecolor='white', linewidth=0.5)
ax.bar(x + w/2, impact['With'],    w, label='With Feature',    color=PURPLE,    edgecolor='white', linewidth=0.5)

for i, row in impact.reset_index().iterrows():
    ax.annotate(f'+{row["Lift"]:.1f}%', xy=(i, max(row['With'], row['Without']) + 1),
                ha='center', color='#a78bfa', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(impact['Feature'])
ax.set_ylabel('D30 Retention (%)')
ax.set_title('Feature Adoption Impact on D30 Retention', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('../data/feature_impact.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

impact

In [ ]:
# Encode categoricals
model_df = df.copy()
le = LabelEncoder()
for col in ['segment','acquisition_channel','device','country','age_group']:
    model_df[col + '_enc'] = le.fit_transform(model_df[col])

feature_cols = [
    'segment_enc','acquisition_channel_enc','device_enc','country_enc','age_group_enc',
    'used_fyp','used_search','used_live','used_shop','used_duet',
    'notification_enabled','total_sessions','total_watch_minutes','videos_posted'
]

X = model_df[feature_cols]
y = model_df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color=PURPLE, linewidth=2, label=f'ROC AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Churn Predictor', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.2)

# Feature importance (coefficients)
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': clf.coef_[0]})
coef_df = coef_df.sort_values('coef', ascending=True)
colors_coef = [PURPLE if c > 0 else '#f87171' for c in coef_df['coef']]
axes[1].barh(coef_df['feature'], coef_df['coef'], color=colors_coef, edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='white', linewidth=0.8, alpha=0.5)
axes[1].set_title('Logistic Regression Coefficients (Churn)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Coefficient')
axes[1].grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig('../data/churn_model.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

In [ ]:
print('=== KEY INSIGHTS ===')
print(f'Overall D30 retention: {df["d30_retained"].mean():.1%}')
print(f'Best segment (D30): {df.groupby("segment")["d30_retained"].mean().idxmax()}')
print(f'Best channel (D30): {df.groupby("acquisition_channel")["d30_retained"].mean().idxmax()}')
print(f'Highest churn segment: {df.groupby("segment")["churned"].mean().idxmax()}')
top_feature = impact.iloc[0]["Feature"]
top_lift    = impact.iloc[0]["Lift"]
print(f'Top feature for retention: {top_feature} (+{top_lift:.1f}% D30 lift)')